# Standard Stratified Random Forest Experiments

## 1. Experiment overview

This notebook reconstructs the project's three existing binary IoT botnet detection experiments without changing their methodology:

1. Random Forest with all 115 traffic features;
2. Random Forest with the 20 highest-importance features from the baseline model;
3. the same Top-20 Random Forest after train-only `StandardScaler` transformation and train-only SMOTE.

The canonical experiment inputs are the existing 80/20 splits in `data/splits/`. The split is row-level and stratified by `binary_target`; it is not temporal or device-held-out. All metrics below are computed from predictions made during notebook execution. Existing files under `outputs/` are read only for validation and are never overwritten.

## 2. Imports and configuration

Paths are resolved by walking upward from the current working directory, so the notebook works when launched from either the repository root or `notebooks/`. The Random Forest configuration includes `verbose=1` because that argument is explicit in every current training script, although it is omitted from the three-parameter configuration shown in the research description. It affects logging, not the fitted estimator.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import platform
import subprocess
import sys
from pathlib import Path
from time import perf_counter

import imblearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
import sklearn
from IPython.display import display
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import StandardScaler


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "workflow.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Start Jupyter from the project or notebooks directory."
    )


PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

TRAIN_PATH = DATA_DIR / "splits" / "all_devices_train_stratified.csv"
TEST_PATH = DATA_DIR / "splits" / "all_devices_test_stratified.csv"
TRAIN_TOP20_PATH = DATA_DIR / "splits" / "all_devices_train_stratified_top20.csv"
TEST_TOP20_PATH = DATA_DIR / "splits" / "all_devices_test_stratified_top20.csv"
FEATURE_IMPORTANCE_REFERENCE_PATH = (
    REPORT_DIR / "all_devices_stratified_random_forest_feature_importance.csv"
)

TARGET_COLUMN = "binary_target"
DROP_COLUMNS = ["binary_label", "binary_target", "source_file"]
CLASS_LABELS = [0, 1]
CLASS_NAMES = ["benign", "attack"]
RANDOM_STATE = 42
RF_CONFIG = {
    "n_estimators": 100,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": 1,
}

EXPECTED_TRAIN_ROWS = 5_650_085
EXPECTED_TEST_ROWS = 1_412_521

required_paths = [
    TRAIN_PATH,
    TEST_PATH,
    TRAIN_TOP20_PATH,
    TEST_TOP20_PATH,
    FEATURE_IMPORTANCE_REFERENCE_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
sns.set_theme(style="whitegrid")

print(f"Project root: {PROJECT_ROOT}")
print(f"Random Forest configuration: {RF_CONFIG}")

## 3. Environment information

Software and platform values are obtained dynamically. On macOS, `system_profiler` is queried when available to identify the computer model and Apple chip.

In [ ]:
def environment_information() -> pd.Series:
    info = {
        "Operating system": platform.system(),
        "OS version": platform.mac_ver()[0] if platform.system() == "Darwin" else platform.release(),
        "Architecture": platform.machine(),
        "Processor": platform.processor() or "not reported",
        "Physical CPU cores": psutil.cpu_count(logical=False),
        "Logical CPU cores": psutil.cpu_count(logical=True),
        "RAM (GiB)": round(psutil.virtual_memory().total / (1024**3), 2),
        "Python": platform.python_version(),
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "imbalanced-learn": imblearn.__version__,
    }

    if platform.system() == "Darwin":
        try:
            completed = subprocess.run(
                ["system_profiler", "SPHardwareDataType", "-json"],
                check=True,
                capture_output=True,
                text=True,
            )
            hardware = json.loads(completed.stdout)["SPHardwareDataType"][0]
            info["Computer model"] = hardware.get("machine_name", "not reported")
            info["Model identifier"] = hardware.get("machine_model", "not reported")
            info["Chip"] = hardware.get("chip_type", "not reported")
        except (OSError, subprocess.SubprocessError, KeyError, IndexError, json.JSONDecodeError):
            info["Computer model"] = "not available from system_profiler"
            info["Chip"] = "not available from system_profiler"
    return pd.Series(info, name="Value")


environment_info = environment_information()
display(environment_info.to_frame())

## 4. Canonical split provenance and header validation

`split_all_devices_stratified.py` creates exact per-class 20% test quotas with `numpy.random.default_rng(42)` while streaming the combined labeled CSV. The training scripts then consume the generated CSVs directly. To preserve that established workflow and avoid rewriting roughly 7 million rows, this notebook loads those canonical split files rather than regenerating them.

The split is stratified globally by `binary_target` only. A device or source file may therefore contribute rows to both train and test. This is an in-distribution row-level evaluation, not cross-device validation.

In [ ]:
def csv_columns(path: Path) -> list[str]:
    return pd.read_csv(path, nrows=0).columns.tolist()


train_columns = csv_columns(TRAIN_PATH)
test_columns = csv_columns(TEST_PATH)
train_top20_columns = csv_columns(TRAIN_TOP20_PATH)
test_top20_columns = csv_columns(TEST_TOP20_PATH)

if train_columns != test_columns:
    raise ValueError("Baseline train/test headers or column order differ.")
if train_top20_columns != test_top20_columns:
    raise ValueError("Top-20 train/test headers or column order differ.")

baseline_feature_columns = [column for column in train_columns if column not in DROP_COLUMNS]
cached_top20_feature_columns = [
    column for column in train_top20_columns if column not in DROP_COLUMNS
]

header_validation = pd.DataFrame(
    [
        {
            "Input": "Baseline train/test",
            "Total columns": len(train_columns),
            "Feature columns": len(baseline_feature_columns),
            "Metadata/label columns": len(train_columns) - len(baseline_feature_columns),
            "Train/test order identical": train_columns == test_columns,
        },
        {
            "Input": "Top-20 train/test",
            "Total columns": len(train_top20_columns),
            "Feature columns": len(cached_top20_feature_columns),
            "Metadata/label columns": len(train_top20_columns) - len(cached_top20_feature_columns),
            "Train/test order identical": train_top20_columns == test_top20_columns,
        },
    ]
)
display(header_validation)

assert len(baseline_feature_columns) == 115
assert len(cached_top20_feature_columns) == 20
assert all(column in train_columns for column in DROP_COLUMNS)
assert all(column in train_top20_columns for column in DROP_COLUMNS)

## 5. Dataset loading and split validation

Only the 115 feature columns and `binary_target` are loaded. Features use `float32` and the target uses `uint8`, matching the training scripts. These in-memory objects are reused for the baseline model and released before the reduced data are loaded.

Label generation is defined by `scripts/export_labeled_devices.py`: files whose traffic type is exactly `benign` map to `binary_label="benign"` and `binary_target=0`; every other traffic file maps to `attack` and `binary_target=1`. Attack (`1`) is the positive class for Precision, Recall, and F1.

In [ ]:
def read_features_target(
    path: Path, feature_columns: list[str]
) -> tuple[pd.DataFrame, pd.Series]:
    dtypes = {column: "float32" for column in feature_columns}
    dtypes[TARGET_COLUMN] = "uint8"
    features = pd.read_csv(
        path,
        usecols=[*feature_columns, TARGET_COLUMN],
        dtype=dtypes,
    )
    target = features.pop(TARGET_COLUMN)
    if features.columns.tolist() != feature_columns:
        raise ValueError(f"Unexpected feature order in {path}")
    return features, target


print("Loading canonical 115-feature training split...")
X_train, y_train = read_features_target(TRAIN_PATH, baseline_feature_columns)
print("Loading canonical 115-feature test split...")
X_test, y_test = read_features_target(TEST_PATH, baseline_feature_columns)

In [ ]:
def class_distribution(target: pd.Series) -> pd.DataFrame:
    counts = target.value_counts().reindex(CLASS_LABELS, fill_value=0).astype("int64")
    return pd.DataFrame(
        {
            "Class": CLASS_NAMES,
            "Count": counts.to_numpy(),
            "Percent": counts.to_numpy() / len(target) * 100,
        },
        index=pd.Index(CLASS_LABELS, name=TARGET_COLUMN),
    )


observed_train_rows = len(X_train)
observed_test_rows = len(X_test)

actual_shapes = pd.DataFrame(
    [
        {
            "Split": "Train",
            "Rows": observed_train_rows,
            "Expected rows": EXPECTED_TRAIN_ROWS,
            "Matches expected": observed_train_rows == EXPECTED_TRAIN_ROWS,
            "Features": X_train.shape[1],
        },
        {
            "Split": "Test",
            "Rows": observed_test_rows,
            "Expected rows": EXPECTED_TEST_ROWS,
            "Matches expected": observed_test_rows == EXPECTED_TEST_ROWS,
            "Features": X_test.shape[1],
        },
    ]
)
display(actual_shapes)

split_distribution = pd.concat(
    {"Train": class_distribution(y_train), "Test": class_distribution(y_test)},
    names=["Split"],
)
display(split_distribution)

assert X_train.columns.tolist() == X_test.columns.tolist()
assert X_train.shape[1] == 115
assert set(pd.unique(y_train)) == {0, 1}
assert set(pd.unique(y_test)) == {0, 1}

## 6. Experiment 1 — Baseline Random Forest (115 features)

The baseline removes only `binary_label`, `binary_target`, and `source_file`; the remaining 115 columns are model features. No scaling, feature elimination, or class balancing is used.

In [ ]:
def evaluate_binary_predictions(
    y_true: pd.Series, y_pred: np.ndarray
) -> tuple[dict[str, float], np.ndarray, str]:
    attack_precision, attack_recall, attack_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )
    benign_precision, benign_recall, benign_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=0,
        zero_division=0,
    )
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Attack Precision": attack_precision,
        "Attack Recall": attack_recall,
        "Attack F1": attack_f1,
        "Benign Precision": benign_precision,
        "Benign Recall": benign_recall,
        "Benign F1": benign_f1,
    }
    matrix = confusion_matrix(y_true, y_pred, labels=CLASS_LABELS)
    report = classification_report(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
    return metrics, matrix, report


experiment_results: list[dict[str, object]] = []
confusion_matrices: dict[str, np.ndarray] = {}

In [ ]:
baseline_model = RandomForestClassifier(**RF_CONFIG)

train_start = perf_counter()
baseline_model.fit(X_train, y_train)
baseline_train_seconds = perf_counter() - train_start

predict_start = perf_counter()
baseline_predictions = baseline_model.predict(X_test)
baseline_predict_seconds = perf_counter() - predict_start

baseline_metrics, baseline_matrix, baseline_report = evaluate_binary_predictions(
    y_test, baseline_predictions
)
baseline_result = {
    "Experiment": "RF Baseline",
    "Features": 115,
    "SMOTE": False,
    "Scaling": "None",
    **baseline_metrics,
    "Train Seconds": baseline_train_seconds,
    "Predict Seconds": baseline_predict_seconds,
}
experiment_results.append(baseline_result)
confusion_matrices["RF Baseline"] = baseline_matrix

display(pd.Series(baseline_result, name="Value").to_frame())
print(baseline_report)

## 7. Feature importance and Top-20 selection

The fitted baseline model's impurity-based `feature_importances_` values are sorted descending, exactly as in `train_all_devices_stratified_baseline_rf.py`. The first 20 names are the selector output. This selector was fit only on the canonical training split; the standard test split did not contribute to feature selection.

`create_all_devices_stratified_top20_splits.py` asks pandas to read those 20 names with `usecols`. Pandas writes them in their order in the original 115-feature CSV, not importance rank order. Consequently, the ranking and cached header order differ, but their feature sets are identical. The cached header order is the exact order consumed by Experiments 2 and 3.

In [ ]:
feature_importance = (
    pd.DataFrame(
        {
            "feature": baseline_feature_columns,
            "importance": baseline_model.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance.index = feature_importance.index + 1
feature_importance.index.name = "rank"
computed_top20_ranked = feature_importance.head(20)["feature"].tolist()

reference_feature_importance = pd.read_csv(FEATURE_IMPORTANCE_REFERENCE_PATH)
reference_top20_ranked = reference_feature_importance.head(20)["feature"].tolist()

top20_validation = pd.Series(
    {
        "Computed Top-20 matches existing ranked list": (
            computed_top20_ranked == reference_top20_ranked
        ),
        "Computed Top-20 set matches canonical reduced CSVs": (
            set(computed_top20_ranked) == set(cached_top20_feature_columns)
        ),
        "Importance rank order equals cached model-input order": (
            computed_top20_ranked == cached_top20_feature_columns
        ),
        "Canonical Top-20 train/test order identical": (
            train_top20_columns == test_top20_columns
        ),
    },
    name="Result",
)
display(top20_validation.to_frame())

if set(computed_top20_ranked) != set(cached_top20_feature_columns):
    raise ValueError(
        "The newly selected Top-20 set differs from the canonical reduced split inputs. "
        "Investigate the existing feature-importance and split artifacts before continuing."
    )

display(feature_importance)

top20_plot = feature_importance.head(20).sort_values("importance")
ax = top20_plot.plot.barh(
    x="feature",
    y="importance",
    legend=False,
    figsize=(9, 7),
    color="#356aa0",
)
ax.set_title("Top-20 Random Forest Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
# Release the 115-feature matrices and baseline estimator before loading reduced inputs.
del baseline_model, baseline_predictions, X_train, X_test, y_train, y_test
gc.collect()

## 8. Experiment 2 — Top-20 Random Forest

The existing canonical Top-20 split CSVs are loaded once and used by both remaining experiments. Their feature order is validated before fitting. No scaling or balancing is used in this experiment.

In [ ]:
print("Loading canonical Top-20 training split...")
X_train_top20, y_train_top20 = read_features_target(
    TRAIN_TOP20_PATH, cached_top20_feature_columns
)
print("Loading canonical Top-20 test split...")
X_test_top20, y_test_top20 = read_features_target(
    TEST_TOP20_PATH, cached_top20_feature_columns
)

assert X_train_top20.columns.tolist() == X_test_top20.columns.tolist()
assert X_train_top20.shape == (observed_train_rows, 20)
assert X_test_top20.shape == (observed_test_rows, 20)

top20_model = RandomForestClassifier(**RF_CONFIG)
train_start = perf_counter()
top20_model.fit(X_train_top20, y_train_top20)
top20_train_seconds = perf_counter() - train_start

predict_start = perf_counter()
top20_predictions = top20_model.predict(X_test_top20)
top20_predict_seconds = perf_counter() - predict_start

top20_metrics, top20_matrix, top20_report = evaluate_binary_predictions(
    y_test_top20, top20_predictions
)
top20_result = {
    "Experiment": "RF Top-20",
    "Features": 20,
    "SMOTE": False,
    "Scaling": "None",
    **top20_metrics,
    "Train Seconds": top20_train_seconds,
    "Predict Seconds": top20_predict_seconds,
}
experiment_results.append(top20_result)
confusion_matrices["RF Top-20"] = top20_matrix

display(pd.Series(top20_result, name="Value").to_frame())
print(top20_report)

del top20_model, top20_predictions
gc.collect()

## 9. Experiment 3 — Top-20 + train-only scaling and SMOTE

The source implementation fits a default `StandardScaler` on `X_train_top20`, transforms train and test with that train-fitted scaler, and then calls `SMOTE(random_state=42, k_neighbors=5)` on the scaled training data only. The test target and test row count remain untouched. Scaling is present only in this experiment; it is not added to the baseline, plain Top-20, or LODO experiments.

In [ ]:
before_smote_distribution = class_distribution(y_train_top20)
test_rows_before_smote = len(X_test_top20)
test_counts_before_smote = y_test_top20.value_counts().sort_index()

scale_start = perf_counter()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_top20).astype("float32", copy=False)
X_test_scaled = scaler.transform(X_test_top20).astype("float32", copy=False)
scale_seconds = perf_counter() - scale_start

# The raw feature frames are no longer needed after the train-fitted transformation.
del X_train_top20, X_test_top20
gc.collect()

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
smote_start = perf_counter()
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train_top20)
smote_seconds = perf_counter() - smote_start
y_train_smote = pd.Series(y_train_smote, name=TARGET_COLUMN)
after_smote_distribution = class_distribution(y_train_smote)

distribution_comparison = pd.concat(
    {
        "Before SMOTE": before_smote_distribution,
        "After SMOTE": after_smote_distribution,
    },
    names=["Stage"],
)
display(distribution_comparison)

assert len(X_test_scaled) == test_rows_before_smote
assert y_test_top20.value_counts().sort_index().equals(test_counts_before_smote)

del X_train_scaled, y_train_top20
gc.collect()

In [ ]:
smote_model = RandomForestClassifier(**RF_CONFIG)
train_start = perf_counter()
smote_model.fit(X_train_smote, y_train_smote)
smote_train_seconds = perf_counter() - train_start

predict_start = perf_counter()
smote_predictions = smote_model.predict(X_test_scaled)
smote_predict_seconds = perf_counter() - predict_start

smote_metrics, smote_matrix, smote_report = evaluate_binary_predictions(
    y_test_top20, smote_predictions
)
smote_result = {
    "Experiment": "RF Top-20 + SMOTE",
    "Features": 20,
    "SMOTE": True,
    "Scaling": "StandardScaler (train-fitted)",
    **smote_metrics,
    "Train Seconds": smote_train_seconds,
    "Predict Seconds": smote_predict_seconds,
}
experiment_results.append(smote_result)
confusion_matrices["RF Top-20 + SMOTE"] = smote_matrix

display(pd.Series(smote_result, name="Value").to_frame())
print(f"Scaling seconds: {scale_seconds:.2f}")
print(f"SMOTE seconds: {smote_seconds:.2f}")
print(smote_report)

del (
    smote_model,
    smote_predictions,
    X_train_smote,
    y_train_smote,
    X_test_scaled,
    y_test_top20,
    scaler,
    smote,
)
gc.collect()

## 10. Experiment comparison

The table is populated only from metrics calculated above. Timing columns measure the Random Forest `fit` and `predict` calls consistently; scaling and SMOTE preparation times are reported separately in Experiment 3.

In [ ]:
comparison_columns = [
    "Experiment",
    "Features",
    "SMOTE",
    "Scaling",
    "Accuracy",
    "Attack Precision",
    "Attack Recall",
    "Attack F1",
    "Train Seconds",
    "Predict Seconds",
]
experiment_comparison = pd.DataFrame(experiment_results)[comparison_columns]
display(experiment_comparison)

## 11. Confusion matrices and read-only reference validation

Each plotted matrix comes from predictions generated in this notebook. The existing CSV matrices are loaded only to check reproducibility; no existing report or model is modified.

In [ ]:
reference_matrix_paths = {
    "RF Baseline": REPORT_DIR / "all_devices_stratified_random_forest_confusion_matrix.csv",
    "RF Top-20": REPORT_DIR / "all_devices_stratified_random_forest_top20_confusion_matrix.csv",
    "RF Top-20 + SMOTE": (
        REPORT_DIR / "all_devices_stratified_random_forest_top20_smote_confusion_matrix.csv"
    ),
}

reference_checks = []
for experiment, matrix in confusion_matrices.items():
    path = reference_matrix_paths[experiment]
    reference_matrix = pd.read_csv(path, index_col=0).to_numpy(dtype="int64")
    reference_checks.append(
        {
            "Experiment": experiment,
            "Reference path": path.relative_to(PROJECT_ROOT),
            "Exact confusion-matrix match": np.array_equal(matrix, reference_matrix),
        }
    )
display(pd.DataFrame(reference_checks))

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
common_vmax = max(int(matrix.max()) for matrix in confusion_matrices.values())
for axis, (experiment, matrix) in zip(axes, confusion_matrices.items()):
    matrix_frame = pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES)
    annotations = matrix_frame.map(lambda value: f"{int(value):,}")
    sns.heatmap(
        matrix_frame,
        annot=annotations,
        fmt="",
        cmap="Blues",
        cbar=False,
        vmin=0,
        vmax=common_vmax,
        square=True,
        ax=axis,
    )
    axis.set_title(experiment)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("Actual class")
plt.show()

In [ ]:
reproducibility_summary = pd.Series(
    {
        "Canonical train split": TRAIN_PATH.relative_to(PROJECT_ROOT),
        "Canonical test split": TEST_PATH.relative_to(PROJECT_ROOT),
        "Train rows observed": observed_train_rows,
        "Test rows observed": observed_test_rows,
        "Target": "binary_target (0=benign, 1=attack)",
        "RF configuration": RF_CONFIG,
        "Feature selection": "Baseline RF importance fitted on canonical train only",
        "SMOTE": "Experiment 3 train only; random_state=42, k_neighbors=5",
        "Scaling": "Experiment 3 only; StandardScaler fitted on train only",
        "Python": platform.python_version(),
        "scikit-learn": sklearn.__version__,
        "imbalanced-learn": imblearn.__version__,
    },
    name="Value",
)
display(reproducibility_summary.to_frame())

## 12. Reproducibility summary

This reconstruction uses the existing canonical split CSVs, `binary_target` as the label (`1` = attack), seed 42 for the custom split provenance, Random Forest, and SMOTE, and the source RF configuration of 100 estimators, all CPU cores, and `verbose=1`. Feature importance is learned from standard training rows only. Experiment 3 alone uses a train-fitted `StandardScaler` followed by train-only SMOTE; the test set is only transformed by that scaler and is never oversampled. The row-level standard split may contain the same devices and source files on both sides and should not be interpreted as unseen-device performance.